# Preprocessing DuckSoup recordings: the one call

All you need to preprocess a DuckSoup session is **one function call**:

```python
from ducksoup import ds_process_parallel
ds_process_parallel(sources="data/my_session/*/recordings/", target_folder="preproc/my_session/")
```

This notebook explains that call: what to give it, what each option does, what you get, and what to check afterwards.

The call:
- puts every participant's video and audio of each interaction on one clock (time 0 = start of the interaction), with audio and video in sync;
- joins reconnections;
- encodes once, at native resolution, visually identical to the recordings (or lossless if you ask);
- writes a report of what it did (`alignment.csv`).

The [README](../README.md) explains how it does that, for those who want the details.

## 1. What `sources` must point to

DuckSoup writes one folder per interaction. The call needs those folders as DuckSoup wrote them:

```
data/my_session/
├── 1-p1p2/
│   ├── recordings/                     <- the videos: i-…-u-p1-c-1-dry.mp4, i-…-u-p1-c-1-wet.mp4, ...
│   └── 1-p1p2-a-20240112-153228.log    <- the DuckSoup log(s) of the interaction: keep them here
├── 1-p3p4/
│   ├── recordings/
│   └── ...
└── ...
```

- `sources` is a pattern (glob) that matches the `recordings/` folders: `"data/my_session/*/recordings/"`. To process several sessions at once, use `"data/*/*/recordings/"`.
- **Keep the `*.log` files next to `recordings/`.** They hold the true recording times, which the alignment is based on. Without them, prepro falls back to the times in the file names (less accurate) and says so in the report.
- Each interaction's outputs go to a folder of the same name (`1-p1p2/`, …) in `target_folder`.

> **Keep your original DuckSoup recordings and logs.** prepro never modifies them, but its outputs are a processed copy (re-encoded, cut at the start of the interaction, reconnections joined). The originals are the only complete record, and you may want to preprocess again later, e.g. with other options or a newer prepro.

## 2. An example session to run on

To have something to run in this notebook, the next cell writes a tiny made-up session with the same layout: one interaction, two participants, dry and wet, a reconnection, and a 70 s recording. **With your own data you skip this cell**; point `sources` at your data instead.

In [1]:
import sys
sys.path.insert(0, "..")                # the prepro folder (not needed if prepro is installed with conda develop)
from example_data import make_example_session

session = make_example_session("example")
sorted(str(p) for p in session.rglob("*") if p.is_file())

['example/data/my_session/1-p1p2/1-p1p2-a-tutorial.log',
 'example/data/my_session/1-p1p2/recordings/i-0123abcd-a-20240101-115952.000-s-tutorial-n-1-p1p2-u-p1-c-1-dry.mp4',
 'example/data/my_session/1-p1p2/recordings/i-0123abcd-a-20240101-115952.000-s-tutorial-n-1-p1p2-u-p1-c-1-wet.mp4',
 'example/data/my_session/1-p1p2/recordings/i-0123abcd-a-20240101-115959.000-s-tutorial-n-1-p1p2-u-p2-c-1-dry.mp4',
 'example/data/my_session/1-p1p2/recordings/i-0123abcd-a-20240101-115959.000-s-tutorial-n-1-p1p2-u-p2-c-1-wet.mp4',
 'example/data/my_session/1-p1p2/recordings/i-0123abcd-a-20240101-120032.000-s-tutorial-n-1-p1p2-u-p2-c-2-dry.mp4',
 'example/data/my_session/1-p1p2/recordings/i-0123abcd-a-20240101-120032.000-s-tutorial-n-1-p1p2-u-p2-c-2-wet.mp4']

## 3. The call

`n_cores=1` because the example is tiny. On your data, leave it out (default: half of your computer's cores) or set it yourself (section 5).

In [2]:
from ducksoup import ds_process_parallel

ds_process_parallel(sources="example/data/my_session/*/recordings/",
                    target_folder="example/preproc/my_session/",
                    n_cores=1, verbose=False)

## 4. What you get

In `target_folder`:

| folder | content |
|---|---|
| `trimed/<interaction>/` | **the files for your analyses**: one `.mp4` and one `.wav` per participant and version (`dry`: the participant's camera and microphone; `wet`: after the effect, as the partner saw it) |
| `trimed/<interaction>/alignment.csv` | the report: one row per output (section 6) |
| `combined/<interaction>.mp4` | the participants side by side, to check by eye |
| `with_audio/<interaction>.mp4` | the same with the sound of everyone |

In every output of an interaction, **time 0 is the start of the interaction**. Frame `k` is at `k / 30` s, and WAV sample `n` is at `n / 48000` s. So the participants can be compared frame by frame.

In [3]:
from pathlib import Path
for p in sorted(Path("example/preproc/my_session").rglob("*")):
    if p.is_file():
        print(f"{str(p):110s} {p.stat().st_size / 1e6:6.1f} MB")

example/preproc/my_session/combined/1-p1p2.mp4                                                                    0.1 MB
example/preproc/my_session/trimed/1-p1p2/alignment.csv                                                            0.0 MB
example/preproc/my_session/trimed/1-p1p2/i-0123abcd-a-20240101-115952.000-s-tutorial-n-1-p1p2-u-p1-c-1-dry.mp4    1.3 MB
example/preproc/my_session/trimed/1-p1p2/i-0123abcd-a-20240101-115952.000-s-tutorial-n-1-p1p2-u-p1-c-1-dry.wav    6.7 MB
example/preproc/my_session/trimed/1-p1p2/i-0123abcd-a-20240101-115952.000-s-tutorial-n-1-p1p2-u-p1-c-1-wet.mp4    1.3 MB
example/preproc/my_session/trimed/1-p1p2/i-0123abcd-a-20240101-115952.000-s-tutorial-n-1-p1p2-u-p1-c-1-wet.wav    6.6 MB
example/preproc/my_session/trimed/1-p1p2/i-0123abcd-a-20240101-115959.000-s-tutorial-n-1-p1p2-u-p2-c-1-dry.mp4    1.3 MB
example/preproc/my_session/trimed/1-p1p2/i-0123abcd-a-20240101-115959.000-s-tutorial-n-1-p1p2-u-p2-c-1-dry.wav    6.7 MB
example/preproc/my_session/trime

## 5. The options

All options have sensible defaults; most users only ever set `n_cores`, and sometimes `duration` or `lossless`.

| option | default | what it does, and when to change it |
|---|---|---|
| `sources` | (required) | the `recordings/` folders to process (section 1) |
| `target_folder` | `"preproc/"` | where the outputs go. It is created if needed |
| `n_cores` | half of the computer's cores | **how many CPU cores the run may use.** The computer stays usable with the default. More cores means more interactions in parallel. On a cluster, give the cores of your job: `n_cores=int(os.environ["SLURM_CPUS_PER_TASK"])` |
| `lossless` | `False` | `False`: x264 CRF 18, visually identical to lossless and about the size of the originals (~60–80 MB per 4-min video); enough for almost all analyses. `True`: no loss added at all (~7x larger), for pixel-level measures such as rPPG. See below |
| `duration` | `None` | cut every output at this many seconds from the start of the interaction, e.g. `duration=240` when the task lasted 4 min. Default: keep everything |
| `target_fps` | `30` | frame rate of the output videos |
| `av_sync` | `True` | measure and correct the audio-video sync of each recording from its content, and check every output. `False` skips the content measurement, which is faster: the offsets then come from the log only, and are flagged as unverified |
| `face_model` | `None` | the MediaPipe face model used to measure the sync from the lips (the most accurate way; needs `pip install mediapipe` and the model file, see the README). Default: the `PREPRO_FACE_MODEL` environment variable, or `models/face_landmarker.task`. Without a model, OpenCV is used |
| `combine_videos` | `True` | also make the side-by-side check videos (`combined/`, `with_audio/`). `False` saves time and space |
| `write_wav` | `True` | also write a lossless `.wav` next to each video (recommended for audio analyses) |
| `verbose` | `True` | print the progress |
| `extension` | `".mp4"` | extension of the DuckSoup recordings |

**Lossless or not?** Usually not: the default (`lossless=False`) is enough.
- **Why:** DuckSoup records the WebRTC streams as the browsers sent them. These are already compressed live, at a modest resolution: mostly 600x450 or 800x600, at ~1–1.5 Mbit/s. When a participant's connection was weak, the browser lowered the resolution during the call (down to 400x300 or 300x225); prepro then scales those parts up to the recording's largest size, which adds no detail.
- **`lossless=False` (default)** re-encodes once at x264 CRF 18. Measured against the lossless version of the same recordings, it is visually identical: SSIM ≈ 0.99 and PSNR 46–50 dB, never below 44 dB, where above ~40 dB is generally invisible. The files are about the size of the originals (~60–80 MB per 4-min recording). It is enough for face landmarks and action units (MediaPipe, py-feat, OpenFace), gestures and body motion, annotation and watching. For an even closer copy, lower `crf_single_pass` (e.g. 12); files grow accordingly.
- **`lossless=True`** adds no loss at all: the frames are exactly those of the recording, in files ~7x larger (~400–600 MB per 4-min recording). Use it for pixel-level measures that depend on tiny colour or brightness changes, such as remote heart rate (rPPG).

**Typical calls:**
```python
# a laptop: everything with the defaults
ds_process_parallel(sources="data/my_session/*/recordings/", target_folder="preproc/my_session/")

# only the first 4 minutes, no check videos
ds_process_parallel(sources="data/my_session/*/recordings/", target_folder="preproc/my_session/",
                    duration=240, combine_videos=False)

# lossless outputs, e.g. for remote heart rate (rPPG)
ds_process_parallel(sources="data/my_session/*/recordings/", target_folder="preproc_lossless/my_session/",
                    lossless=True)

# all sessions at once, with the MediaPipe sync measurement, on 16 cores
ds_process_parallel(sources="data/*/*/recordings/", target_folder="preproc/", n_cores=16,
                    face_model="models/face_landmarker.task")
```

Running the call again on the same `target_folder` skips the outputs that already exist, so an interrupted run can be resumed.
The full list, including a few advanced options, is in `help(ds_process_parallel)`.

## 6. Checking the run: `alignment.csv`

Each interaction folder has a report with one row per output. Read it after a run. The columns to look at:

| column | look for |
|---|---|
| `status` | `ok`. Otherwise: `skipped_empty_file` (DuckSoup wrote an empty recording), `skipped_before_t0`, or `ffmpeg_error: …` |
| `duration_s`, `covered_s`, `n_gaps`, `gaps_s` | how much of the interaction was recorded, and the gaps from reconnections (black frames and silence in the output: mask them in analyses) |
| `audio_offset_s`, `video_offset_s` | the corrections applied: audio that was late in the recording, and a wet video moved onto its dry twin |
| `offset_source` | how the audio correction was decided. `duration+content` / `content`: verified. `duration`: unverified |
| `av_check` | the audio-video sync of the finished video, measured again: `ok`, `inconclusive` (too little visible talking to tell), `mismatch` (look at it) |
| `notes` | anything worth a look |

In [4]:
import pandas as pd
report = pd.read_csv("example/preproc/my_session/trimed/1-p1p2/alignment.csv")
report[["user", "manip", "status", "duration_s", "covered_s", "n_gaps", "gaps_s",
        "audio_offset_s", "video_offset_s", "offset_source", "av_check", "notes"]]

,user,manip,status,duration_s,covered_s,n_gaps,gaps_s,audio_offset_s,video_offset_s,offset_source,av_check,notes
0,p1,dry,ok,70.0,70.0,0,0.0,2.0,0.0,duration+content,ok,NaN
1,p1,wet,ok,69.0,69.0,0,0.0,2.0,-1.0,dry twin: duration+content,ok,c1: audio from the dry twin
2,p2,dry,ok,70.0,66.0,1,4.0,0.0,0.0,"none,content",ok,NaN
3,p2,wet,ok,70.0,66.0,1,4.0,0.0,0.0,"dry twin: none,dry twin: content",ok,c1: audio from the dry twin; c2: audio from th...


In the example:
- p1's audio was 2 s late and was corrected, with the correction verified (`duration+content`).
- p1's wet video was moved 1 s onto its dry twin.
- p2's reconnection left a 4 s gap.
- Every finished video passed the sync check.

## 7. On a computing cluster

For a large dataset, run one job per session, for example with Slurm:

```bash
#!/bin/bash
#SBATCH --cpus-per-task=16
#SBATCH --mem=32G
#SBATCH --time=0-10:00:00
#SBATCH --array=0-9                         # one task per session
python -c "
import os, glob
from ducksoup import ds_process_parallel
s = sorted(glob.glob('data/*/'))[int(os.environ['SLURM_ARRAY_TASK_ID'])]
ds_process_parallel(sources=s + '*/recordings/', target_folder='preproc/' + os.path.basename(s.rstrip('/')) + '/',
                    n_cores=int(os.environ['SLURM_CPUS_PER_TASK']))
"
```

**Time:** with MediaPipe, a 4-min recording takes about 3–5 min of CPU (sync measurement, lossless encode, check).